# 07 — Land Masking Demo

Demonstrates `utils/masking.py` on a sample of tiles.

For each selected tile, shows:
- Original GeoTIFF (RGB composite)
- Masked output (same tile after land removal, NaN pixels shown in red)

Run this to verify masking before committing to a full preprocessing run.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PLANET_ROOT   = "/path/to/planet_superdove"           # raw (unmasked) tiles
LANDMASK_ROOT = "/path/to/land_mask"                  # shapefile directory
OUT_ROOT      = "/path/to/planet_superdove_landmasked" # masked output
FIGURES_DIR   = "figures"

# Sites and number of example tiles to show per site
DEMO_SITES = ["northpoint_lizard", "pulau_kapas", "chachacual_mexico"]
N_TILES_PER_SITE = 2

In [ ]:
import os, sys, glob
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))

from utils.masking import find_landmask_shp, SITE_ALIASES
from utils.viz import to_display_rgb, percentile_stretch

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# ── Verify shapefile lookup for demo sites ────────────────────────────────────
print("Site alias table:", SITE_ALIASES)
print()
for site in DEMO_SITES:
    shp = find_landmask_shp(site, LANDMASK_ROOT)
    status = os.path.basename(os.path.dirname(shp)) if shp else "NOT FOUND"
    print(f"  {site:30s} → {status}")

In [ ]:
# ── Before / after visualization ──────────────────────────────────────────────
def read_rgb(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    return percentile_stretch(to_display_rgb(torch.from_numpy(arr)))

def nan_overlay(path):
    """Return [H,W] boolean mask where any band is NaN."""
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    return np.isnan(arr).any(axis=0)

for site in DEMO_SITES:
    raw_tiles = sorted(glob.glob(
        os.path.join(PLANET_ROOT, site, "**", "loc*.tif"), recursive=True
    ))[:N_TILES_PER_SITE]

    if not raw_tiles:
        print(f"[{site}] No tiles found under {PLANET_ROOT}/{site}")
        continue

    n = len(raw_tiles)
    fig, axes = plt.subplots(n, 2, figsize=(8, n * 3.5))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(site, fontsize=12)

    for row, raw_path in enumerate(raw_tiles):
        rel = os.path.relpath(raw_path, PLANET_ROOT)
        masked_path = os.path.join(OUT_ROOT, rel)

        rgb_raw = read_rgb(raw_path)
        axes[row, 0].imshow(rgb_raw)
        axes[row, 0].set_title(f"Original\n{os.path.basename(raw_path)}")
        axes[row, 0].axis("off")

        if not os.path.exists(masked_path):
            axes[row, 1].text(0.5, 0.5, "masked file not found",
                              ha="center", va="center", transform=axes[row, 1].transAxes)
            axes[row, 1].axis("off")
            continue

        rgb_masked = read_rgb(masked_path)
        nan_mask   = nan_overlay(masked_path)

        axes[row, 1].imshow(rgb_masked)
        # overlay NaN (land) pixels in red
        overlay = np.zeros((*nan_mask.shape, 4), dtype=np.float32)
        overlay[nan_mask] = [1.0, 0.0, 0.0, 0.4]
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(f"Masked (land=red)\n{nan_mask.sum()} NaN px")
        axes[row, 1].axis("off")

    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"masking_demo_{site}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── (Optional) Run full masking pipeline ─────────────────────────────────────
# Uncomment to process all sites. This writes to OUT_ROOT.
# Expect ~10-30 min depending on tile count and disk speed.

# from utils.masking import apply_landmasks
# summary = apply_landmasks(
#     planet_root=PLANET_ROOT,
#     landmask_root=LANDMASK_ROOT,
#     out_root=OUT_ROOT,
#     invert=True,
#     copy_if_no_mask=True,
# )
# for site, counts in summary.items():
#     print(f"{site:30s}  {counts}")